In [1]:
# filed detection
import cv2
# from imageio.core.imopen import imopen
from numpy.ma.core import shape
import numpy as np

# img = cv2.imread('../tmp/v5_bacground.png', cv2.IMREAD_COLOR)  # road.png is the filename

input_video_path = "../materials_part1/2024-11-18 17-53-16.mkv"#"../2024-11-18 17-53-16.mkv"  #"F:\\Videos\\2024-11-18 17-59-05.avi"  #"F:\\Videos\\2024-11-18 17-49-57.mkv"  #"F:\\Videos\\2024-11-18 17-59-05.avi"

cap = cv2.VideoCapture(input_video_path)

from cut_image import get_cut_frame_from_frame

ret, image = cap.read()
for _ in range(100):
    ret, image = cap.read()
    if not ret:
        pass

image = get_cut_frame_from_frame(image, 2)
image = cv2.rotate(image, cv2.ROTATE_180)
img_tmp = cv2.blur(image, (3, 3))
cv2.imwrite("../tmp/hehe4.png", img_tmp)
from sklearn.cluster import KMeans


# flat = img.flatten()
def multiply_list(data):
    tmp = 1
    for i in data:
        tmp *= i
    return tmp


def cluster_and_recolor(img, n_clusters, mode: int = 1, top_n=None, binary=False):
    flat = img.reshape((multiply_list(img.shape) // 3, 3))

    kmeans = KMeans(n_clusters=n_clusters, random_state=0, n_init=3).fit(flat)

    cluster_color_centers = list()
    for i in kmeans.cluster_centers_:
        clr = list()
        for j in i:
            clr.append(int(j))
        cluster_color_centers.append(clr)
    cluster_color_centers = np.array(cluster_color_centers)
    # print(img, "hehe", flat, type(img), "hehe", cluster_color_centers)

    unflat = kmeans.labels_.reshape(img.shape[:-1]).astype(np.uint8)

    if mode == 1:
        return unflat * (255 // n_clusters)
    elif mode == 2 or mode == 3:
        sizes = [0] * len(cluster_color_centers)
        for i in kmeans.labels_:
            sizes[i] += 1
        tmp_sizes = list()
        for num, i in enumerate(sizes):
            tmp_sizes.append((-i, num))
        tmp_sizes = sorted(tmp_sizes)
        remap = [0] * len(sizes)
        for num, i in enumerate(tmp_sizes):
            remap[i[1]] = num
        if mode == 3:
            start_clr = cluster_color_centers[tmp_sizes[0][1]]
            size = tmp_sizes[0][0]
            if top_n is None:
                top_n = 0
            seen = [tmp_sizes[0][1]]
            for i in range(top_n):
                nearest = 0
                dist = np.inf
                size_tmp = 0
                for j in range(n_clusters):
                    if tmp_sizes[j][1] not in seen and np.linalg.norm(
                            start_clr - cluster_color_centers[tmp_sizes[j][1]]) < dist:
                        dist = np.linalg.norm(start_clr - cluster_color_centers[tmp_sizes[j][1]])
                        nearest = tmp_sizes[j][1]
                        size_tmp = tmp_sizes[j][0]
                seen.append(nearest)
                #start_clr = size * (start_clr / (size_tmp + size)) + cluster_color_centers[nearest] * (size_tmp / (size_tmp + size))
                size += size_tmp
            for j in seen:
                remap[j] = seen[0]
        for i in range(len(unflat)):
            for j in range(len(unflat[i])):
                if top_n is None or (mode == 3 and not binary):
                    unflat[i][j] = remap[unflat[i][j]]
                elif binary and mode == 3:
                    if remap[unflat[i][j]] == remap[tmp_sizes[0][1]]:
                        unflat[i][j] = 255
                    else:
                        unflat[i][j] = 0
                elif top_n > remap[unflat[i][j]]:
                    unflat[i][j] = 255
                else:
                    unflat[i][j] = 0
        if top_n is None or mode == 3:
            return unflat * (255 // n_clusters)
        else:
            return unflat
    elif mode == 4:
        for i in range(img.shape[0]):
            for j in range(img.shape[1]):
                img[i][j] = cluster_color_centers[unflat[i][j]]
        return img


# image = cv2.blur(image, (3, 3))
# image = cv2.blur(image, (3, 3))
# img = cluster_and_recolor(image, 13, 3, top_n=2, binary=True)  #13 3 2 // 30 3 6 // 20 3 3
# print(img)
# img = cluster_and_recolor(img,4)
# cv2.imshow('lanes', img)
# cv2.waitKey(0)
# cv2.destroyAllWindows()

In [1]:
#lines detection
from image_tuner import rgb_tune_test, tune_from_params, rgb_tune_from_params
import numpy as np
import math
import sklearn.cluster as skcl
from scipy.spatial import ConvexHull
from sklearn.linear_model import LinearRegression

import cv2

# white_color_img = tune_test("../tmp/hehe4.png")  #69 0 153 191 248 255
img_tmp = cv2.imread("../tmp/hehe4.png")
# img_tmp = cv2.imread("/root/Downloads/corridor_field.jpg")
green_color_img = cv2.blur(img_tmp, (3, 3))
green_color_img = cluster_and_recolor(green_color_img, 12, 3, top_n=2, binary=True)  #13 3 2 // 30 3 6 // 20 3 3
# ##########################################################
# Загрузка изображения
image = green_color_img

# Бинаризация изображения
_, binary = cv2.threshold(image, 127, 255, cv2.THRESH_BINARY)

# Поиск контуров
contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# Создание маски
mask = np.zeros_like(image)

# Пороги для фильтрации
aspect_ratio_min = 0.2  # Минимальное аспектное соотношение
aspect_ratio_max = 5.0  # Максимальное аспектное соотношение
area_min = 100          # Минимальная площадь

# Обработка контуров
for contour in contours:
    # Получаем размеры ограничивающего прямоугольника
    x, y, w, h = cv2.boundingRect(contour)

    # Вычисляем аспектное соотношение
    aspect_ratio = float(w) / h if h != 0 else 0

    # Вычисляем площадь
    area = cv2.contourArea(contour)

    # Если это прямоугольник, заливаем его
    if aspect_ratio_min < aspect_ratio < aspect_ratio_max and area > area_min:
        cv2.drawContours(mask, [contour], -1, 255, -1)

# Применяем маску к изображению
result = image.copy()
result[mask == 255] = 255
green_color_img = result
# ##########################################################
# cv2.imshow('ss', green_color_img)
# cv2.waitKey(0)
# cv2.destroyAllWindows()
white_color_img = cv2.blur(img_tmp, (5, 5))
white_color_img = tune_from_params(white_color_img, 33, 0, 143, 181, 76, 255)
cv2.imshow('ss', white_color_img)
cv2.waitKey(0)
cv2.destroyAllWindows()

# ##########################################################
# Загрузка изображения
image = white_color_img

# Бинаризация изображения
_, binary = cv2.threshold(image, 127, 255, cv2.THRESH_BINARY)

# Поиск контуров
contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# Создание маски
mask = np.zeros_like(image)

# Пороги для фильтрации
aspect_ratio_min = 0.2  # Минимальное аспектное соотношение
aspect_ratio_max = 5.0  # Максимальное аспектное соотношение
area_min = 100          # Минимальная площадь

# Обработка контуров
for contour in contours:
    # Получаем размеры ограничивающего прямоугольника
    x, y, w, h = cv2.boundingRect(contour)

    # Вычисляем аспектное соотношение
    aspect_ratio = float(w) / h if h != 0 else 0

    # Вычисляем площадь
    area = cv2.contourArea(contour)

    # Если это прямоугольник, заливаем его
    if aspect_ratio_min < aspect_ratio < aspect_ratio_max and area > area_min:
        cv2.drawContours(mask, [contour], -1, 255, -1)

# Применяем маску к изображению
result = image.copy()
result[mask == 255] = 255
white_color_img = result
# cv2.imshow('ss', white_color_img)
# cv2.waitKey(0)
# cv2.destroyAllWindows()
# rgb_tune_test("/root/Downloads/corridor_field.jpg")
# green_color_img = cluster_and_recolor(green_color_img, 13, 3, top_n=2, binary=True)  #13 3 2 // 30 3 6 // 20 3 3
# white_color_img = tune_from_params(img_tmp, 33, 0, 143, 181, 76, 255)
# green_color_img = tune_from_params(img_tmp,23,113,58,36,255,154)
# white_color_img = rgb_tune_from_params(img_tmp,57,104,117,117,124,137)
# print(white_color_img.shape,len(white_color_img),len(white_color_img[0]))
# cv2.imshow('lanes', white_color_img)
# cv2.waitKey(0)
# cv2.destroyAllWindows()
# print(img)
result_edges = white_color_img * 0
print()
for i in range(1, len(white_color_img) - 1):
    for j in range(1, len(white_color_img[i]) - 1):
        seen_white = 0
        seen_green = 0
        bad = 0
        for k in (-1, 0, 1):
            for t in (-1, 0, 1):
                if green_color_img[i + k][j + t] > 200:
                    seen_green += 1
                if white_color_img[i + k][j + t] > 200:
                    seen_white += 1
                if green_color_img[i + k][j + t] <= 200 and white_color_img[i + k][j + t] <= 200:
                    bad += 1
        if seen_white > 1 and seen_green > 2:
            # print("lane")
            result_edges[i][j] = 255

analysis = cv2.connectedComponentsWithStats(result_edges,
                                            4,
                                            cv2.CV_32S)
(totalLabels, label_ids, values, centroid) = analysis
areas = list()
for i in values:
    areas.append(i[cv2.CC_STAT_AREA])
areas = sorted(areas, reverse=True)
output = np.zeros(result_edges.shape, dtype="uint8")
for i in range(1, totalLabels):
    area = values[i, cv2.CC_STAT_AREA]

    if area >= areas[30]:
        # Labels stores all the IDs of the components on the each pixel
        # It has the same dimension as the threshold
        # So we'll check the component
        # then convert it to 255 value to mark it white
        componentMask = (label_ids == i).astype("uint8") * 255

        # Creating the Final output mask
        output = cv2.bitwise_or(output, componentMask)
    # print (analysis)

# cv2.imshow('lanes', output)
# cv2.waitKey(0)
# cv2.destroyAllWindows()

segments = cv2.HoughLinesP(output, 1, math.pi / 180, 80, 30, 40)
# print(segments.shape,segments)
segments = segments.reshape((len(segments), 2, 2))

# segments[i] = sorted(segments[i])
lines = list()
if False:  #clusterization method   #todo redo segments from vec<int,4> to vec<vec<int,2>,2>
    segm_by_direction_and_sp = list()
    for i_tmp in segments:
        i = i_tmp[0]
        x = i[2] - i[0]
        y = i[3] - i[1]
        if True:
            len_xy = math.sqrt(x * x + y * y)
            x /= len_xy
            y /= len_xy
        if x < 0:
            x *= -1
            y *= -1
        zero = (0, 0)
        if abs(x) > abs(y):
            c1 = i[0] / x
            zero = (0, i[1] - y * c1)
        else:
            c1 = i[1] / y
            zero = (i[0] - x * c1, 0)
        segm_by_direction_and_sp.append((x * 1000, y * 1000, zero[0], zero[1]))
    segm_by_direction_and_sp = np.array(segm_by_direction_and_sp)
    # clusters = skcl.OPTICS(min_samples=4).fit(segm_by_direction_and_sp)
    # clusters = skcl.DBSCAN(min_samples=2).fit(segm_by_direction_and_sp)
    clusters = skcl.HDBSCAN(min_samples=2).fit(segm_by_direction_and_sp)
    # clusters = KMeans(n_clusters=10).fit(segm_by_direction_and_sp)
    centers_of_clusters = dict()
    for i in range(len(segm_by_direction_and_sp)):
        if clusters.labels_[i] not in centers_of_clusters.keys():
            centers_of_clusters[clusters.labels_[i]] = [np.linalg.norm(segments[i][0][:1] - segments[i][0][2:])]
            # print(type(segm_by_direction_and_sp[i].tolist()),,centers_of_clusters[clusters.labels_[i]])
            centers_of_clusters[clusters.labels_[i]].extend(
                (segm_by_direction_and_sp[i] * np.linalg.norm(segments[i][0][:1] - segments[i][0][2:])).tolist())
        else:
            centers_of_clusters[clusters.labels_[i]][0] += np.linalg.norm(segments[i][0][:1] - segments[i][0][2:])
            for j in range(len(segm_by_direction_and_sp[i])):
                centers_of_clusters[clusters.labels_[i]][j + 1] += segm_by_direction_and_sp[i][j] * np.linalg.norm(
                    segments[i][0][:1] - segments[i][0][2:])
    for i in centers_of_clusters.values():
        # print(i)
        # i = [1]
        # i.extend(i_tmp)
        lines.append([[int((i[-2] - 1000 * i[1]) / i[0]), int((i[-1] - 1000 * i[2]) / i[0]),
                       int((i[-2] + 1000 * i[1]) / i[0]), int((i[-1] + 1000 * i[2]) / i[0])]])

#idea: if 2 segments are on line then area on 4 points is minimal


if False:
    for i in range(len(segments)):  #sorted so segms are from x1 to x2 where x1<x2 or x1==x2 and y1<y2
        if segments[i][0][0] < segments[i][1][0]:
            segments[i] = np.array([segments[i][1], segments[i][0]])
        elif segments[i][0][0] == segments[i][1][0] and segments[i][0][1] < segments[i][1][1]:
            segments[i] = np.array([segments[i][1], segments[i][0]])

    st = set()
    for i in segments:
        st.add(tuple(i.reshape((4)).tolist()))
    while True:
        # concatinating segments by using best start and best end points while area metric allows it
        first_best = second_best = None
        best_area = 0
        for i_tmp in st:
            for j_tmp in st:
                i = np.array(i_tmp)
                j = np.array(j_tmp)
                if (i != j).any():
                    area = 0
                    points = np.array([i, j]).reshape((4, 2))
                    for i in range(len(points)):
                        area += np.cross(points[i], points[(i + 1) % len(points)])

                    if first_best is None:
                        first_best = i
                        second_best = j
                        best_area = area
                    else:
                        if best_area > area:
                            best_area = area
                            first_best = i
                            second_best = j
        if area > 500:
            break

        points = np.array([first_best, second_best]).reshape((2, 4))
        line = LinearRegression().fit(points[0], points[1])
        start = (0, line.intercept_)
        direction = (1, line.coef_)


        def project_point(point, start_line, direction_line):
            point -= start_line
            sk = 0
            len_line = 0
            for i in range(len(line)):
                len_line += direction_line[i] ** 2
                sk += direction_line[i] * point[i]
            sk /= len_line
            return start_line + direction_line * sk


        result = (0, 0)
        if first_best[0] < second_best[0]:
            result[0] = first_best[0]
        else:
            result[0] = second_best[0]
        if first_best[0] > second_best[0]:
            result[1] = first_best[1]
        else:
            result[1] = second_best[1]
        result[0] = project_point(result[0], start, direction)
        result[1] = project_point(result[1], start, direction)
        st.remove(first_best)
        st.remove(second_best)
        st.add(np.array(result))
    lines = list(st)

#idea 3: same as idea 2, but using this metric in dbscan
labels = 0
n_clusters = 0
if True:

    def custom_distance_v1(a: np.array, b: np.array):  # works iff a and b goes into different directions
        a = a.reshape((2, 2))
        b = b.reshape((2, 2))
        points = [a[0], a[1], b[0], b[1]]
        sum = np.cross(points[-1], points[0])
        for i in range(1, len(points)):
            sum += np.cross(points[i - 1], points[i])
        return np.linalg.norm(sum)  #eps = 10


    def custom_distance_v2(a: np.array, b: np.array):
        a = a.reshape((2, 2))
        b = b.reshape((2, 2))
        points = [a[0], a[1], b[0], b[1]]
        sum = np.cross(points[-1], points[0])
        for i in range(1, len(points)):
            sum += np.cross(points[i - 1], points[i])
        points[0], points[1] = points[1], points[0]
        sum2 = np.cross(points[-1], points[0])
        for i in range(1, len(points)):
            sum2 += np.cross(points[i - 1], points[i])
        return max(np.linalg.norm(sum), np.linalg.norm(sum2))  #eps = 100


    def custom_distance_v3(a: np.array, b: np.array):
        a = a.reshape((2, 2))
        b = b.reshape((2, 2))
        points = [a[0], a[1], b[0], b[1]]
        sum = np.cross(points[-1], points[0])
        for i in range(1, len(points)):
            sum += np.cross(points[i - 1], points[i])
        points[0], points[1] = points[1], points[0]
        sum2 = np.cross(points[-1], points[0])
        for i in range(1, len(points)):
            sum2 += np.cross(points[i - 1], points[i])
        return max(np.linalg.norm(sum), np.linalg.norm(sum2)) / (
                    np.linalg.norm(a[0] - a[1]) + np.linalg.norm(b[0] - b[1]))  #eps = 5


    # print(segments)
    clustering = skcl.DBSCAN(eps=5, min_samples=1, metric=custom_distance_v3).fit(
        segments.reshape((segments.shape[0], 4)))
    labels = clustering.labels_
    n_clusters = len(np.unique(labels)) - (1 if -1 in labels else 0)
    # print("labels,cluster_n ",labels,n_clusters)


def add_up_segments_v1(data: np.array):
    data = data.astype(np.float64)

    #prepare
    # def is_same_direction(a,b):
    #     v1 = np.cross(b-a[0],a[1]-a[0])/np.linalg.norm(a[1]-a[0])*(a[1]-a[0]).reshape((1,2))
    #     delta = np.linalg.norm((a[1]-a[0])-(v1[1]-v1[0]))
    #     return delta<1e-5
    # def is_same_direction_v2(a,b):
    #     v1 = np.cross(b-a[0],a[1]-a[0])/np.linalg.norm(a[1]-a[0])
    #     delta = 1-(v1[1]-v1[0])
    #     return delta<1e-5
    def is_same_direction_v3(a, b):
        return np.dot(a[1] - a[0], b[1] - b[0]) > 0

    # print(is_same_direction_v3(np.array([[1,2],[3,4]]),np.array([[9,10],[7,8]])))#false
    # print(is_same_direction_v3(np.array([[1,2],[3,4]]),np.array([[5,6],[7,8]])))#true
    for i in range(1, len(data)):
        if not is_same_direction_v3(data[0], data[i]):
            data[i][0], data[i][1] = data[i][1], data[i][0]
    sum_weight = np.linalg.norm(data[0][1] - data[0][0])
    sum_direction = data[0][1] - data[0][0]
    avg_point = (data[0][0] + data[0][1]) * sum_weight
    for i in range(1, len(data)):
        tmp_w = np.linalg.norm(data[i][1] - data[i][0])
        sum_direction += data[i][1] - data[i][0]
        avg_point += (data[i][0] + data[i][1]) * tmp_w
        sum_weight += tmp_w
    sum_direction /= sum_weight
    avg_point /= sum_weight * 2
    start = 0
    end = 0
    for i in range(len(data)):
        start = min(start, np.dot(sum_direction, data[i][0] - avg_point))
        end = max(end, np.dot(sum_direction, data[i][1] - avg_point))
    return (np.array([start * sum_direction, end * sum_direction]) + avg_point).astype(np.int32)


grouped = dict()
for i in range(len(segments)):
    if labels[i] not in grouped.keys():
        grouped[labels[i]] = [segments[i]]
    else:
        grouped[labels[i]].append(segments[i])
if True:
    new_segments = list()
    for i in grouped.values():
        new_segments.append(add_up_segments_v1(np.array(i)))
    segments = new_segments


NameError: name 'cluster_and_recolor' is not defined

In [14]:


def homography_matrix(segs):
    def homography_matrix_by_4_vec(vecs):  # vecs = [a,b,c,d],  a,b -> (1,0) c,d ->(0,1)
        for i in range(len(vecs)):
            vecs[i] = np.append(vecs[i], 1)
        # notice that a-b = alpha is eigenvector with value 0 as well as c-d = beta
        # than:
        # A = V*L*V^(-1)
        # L = [[0,0,0],[0,0,0],[0,0,x]]
        # and V = [alpha,beta,alpha cross beta]
        alpha = vecs[0] - vecs[1]
        beta = vecs[2] - vecs[3]
        gamma = np.cross(alpha, beta)

    # idea v2: try all matricies and choose one which maximizes metric.
    # for all  segments generate vectors length of 1
    # for all pairs of this vectors find matrix that minimizes some function
    vectors = list()
    for i in segs:
        point = (i[0] - i[1]) / np.linalg.norm(i[0] - i[1])
        vectors.append(point)
    vectors_v2 = np.array(vectors).reshape((len(vectors), 1, 2))
    # def get_metric(vecs,)
    best_val = -1e9
    best_m = np.identity(3)
    for i in range(len(vectors)):
        for j in range(len(vectors)):
            if i != j:
                for k in range(len(vectors)):
                    if i != k and j != k:
                        for f in range(len(vectors)):
                            if f != k and f != j and f != k:
                                mat, mask = cv2.findHomography(
                                    np.array([vectors[f], vectors[i], vectors[j], vectors[k]]),
                                    np.array([[0, 100], [100, 0], [100, 0], [0, 100]]))
                                # print(vectors.shape,mat.shape)
                                vecs = cv2.perspectiveTransform(vectors_v2, mat).reshape((len(vectors), 2))
                                groups_size = [0, 0, 0]  # hor,vert,bad
                                for t in vecs:
                                    t_p = t / np.linalg.norm(t)
                                    if np.abs(np.abs(np.dot(t_p, np.array([1, 0]))) - 1) < 1 / 8:
                                        groups_size[0] += 1
                                    elif np.abs(np.abs(np.dot(t_p, np.array([0, 1]))) - 1) < 1 / 8:
                                        groups_size[1] += 1
                                    else:
                                        groups_size[2] += 1
                                val = groups_size[0] * groups_size[1] - groups_size[2]
                                # print(groups_size)
                                if val > best_val:
                                    print(groups_size)
                                    # print()
                                    best_val = val
                                    best_m = mat

    print(best_val, best_m)

    return best_m



# homograpy matrix, for transfromation that minimizes sum of distances between each end of segments and line that goes trough center of segment and "horizontal" or "vertical" point at the horizon, where this points represents points to which are all parallel horizontal/vertical lines are converging
def homography_matrix_v2(hor_point1,hor_point2,center_pos):#,fov_px,fov_angle):
    #hereinafter assuming center of image is (0,0), i.e. (0,0) is point through perpendicular from "focal point" is goes
    #todo:convert edges to p1 and p2 solution: watch in deepseek, has intersting way of searching with ransak. also points p1 and p2 are called vanishing points and there are some papers in internet about them.
    def find_distance_fp_to_plane_v1(fov_pixels,
                                     field_of_view):  #hor[0]*x+hor[1]*y = hor[2] ; field_of_view in degrees - angle at which edges of image displayed, fov_pixels is number of pixels corresponding to fov
        return fov_pixels / (2 * math.tan(field_of_view / 2))

    # do not work if horizontal lines are parallel to horizon.
    #assuming that perpendicular line from "focal point" (point from which rays are shoot in point-and-plane approximation of camera) onto "focal plane" (plane of image in approximation ...) goes through center of image
    def find_distance_fp_to_plane_v2(p1, p2, p3,
                                     angle=math.pi / 2) -> float:  #p1 is horizont point for vertical lines,p2 is horizont point for horizontal lines, p3 is nearest point on horizon to center of image. returns distance in pixels
        #let's derive formula:
        # look on triangle FP, p1,p2:
        # let segment p1-p3 called a, p3-p2 called b, angle fp-p1-p2 is beta, angle p2-fp-p1 is alpha, segment fp-p3 is h
        # angle fp-p3-p1 is 90 deg based on assumption
        # then tg(beta) = h/a, tg(180-alpha-beta) = -tg(alpha + beta) = h/b
        #then -tg(beta)/tg(alpha+beta) = b/a
        #then -tg(beta)/((tg(alpha)+tg(beta))/(tg(1-tg(alpha)tg(beta))) = b/a
        #then tg(beta) = x, tg(alpha) = c
        # (-x+c*x**2)/(x+c) = b/a
        #then -x+c*x**2 -x*b/a-c*b/a = 0
        # same as: c*x**2 +(-1-b/a)*x-c*b/a = 0
        # d = 1+2*b/a+(b/a)**2+4*c**2*b/a
        # x = ...: x is positive
        a = np.linalg.norm(p1 - p3)
        b = np.linalg.norm(p2 - p3)
        c_p = math.tan(
            math.pi / 2 - angle)  # c_p = ctg(angle) = tg(90-angle) = 1/tg(angle) needed because tg(90) -> inf
        # d = 1+2*b/a+(b/a)**2+4*b/a*c*c
        # x = ((1+b/a)+math.sqrt(d))/(2*c)
        h = 0
        if angle != math.pi/2:
            x = ((1 + b / a) * c_p + math.sqrt((1 + 2 * b / a + (b / a) ** 2) * c_p ** 2 + 4 * b / a)) / 2
            h = x * a
        else:
            h = math.sqrt(a*b)#len of perpendicular to hypotenuse is sqrt of multiplication of projections of sides onto hypotenuse

        print("h and p3" ,h,(np.linalg.norm(p3)),(p3))
        ans = math.sqrt(h**2-np.linalg.norm(p3)**2)#h is distance from p3 to focal point, because this line is not necessary perpendicular to focal plane and line that goes through center is, this step is needd
        return ans

    def calculate_p3(p1, p2):
        return np.dot(p1,p1-p2)/np.linalg.norm(p1-p2)**2*(p2-p1)+p1 # simplification of following


        # a = np.linalg.norm(p1)
        # b = np.linalg.norm(p2)
        # c = np.linalg.norm(p1-p2)

        a_p = np.dot(p1, p2 - p1)  #same in case of relation as: np.dot(-p1,p2-p1)/c# same as: (np.dot(-p1,p2-p1)/a/c)*a
        b_p = np.dot(p2, p1 - p2)  #same in case of relation as: np.dot(-p2,p1-p2)/c# same as: (np.dot(-p2,p1-p2)/b/c)*b
        return a_p / (a_p + b_p) * p1 + b_p / (a_p + b_p) * p2

    def angle_of_camera_v1(distance_to_fp, p3):
        """
        :return: angle between camera axis and horisontal line (if camera look at the horison, angle is 0)
        """
        return math.atan(np.linalg.norm(p3) / distance_to_fp)

    def homography_matrix_internal(p1, angle, scale, p3, distance_to_fp):
        """
        assuming camera axis lower than horizon :todo fix this
        :param p1:  is horizont point for vertical lines
        :param angle: angle of tilt perpendicular to horizon
        :param scale: relation of real field(in pixels) to pixels on image
        :param p3: p3
        :param distance_to_fp: one extensive parameter, distance to focal point
        :return: homography matrix
        """
        hdir = np.array([-p3[1], p3[0]])
        hdir /= np.linalg.norm(hdir)
        hp1 = hdir * 1000
        hp2 = -hdir * 1000
        hpr1 = hp1 * scale
        hpr2 = hp2 * scale
        #we dont want to accidentaly put one of point on horizon so we halfing angle between p3 and camera axis
        hp3 = p3 * math.tan(angle/2)/math.tan(angle)
        p3_normed = p3/np.linalg.norm(p3)
        hpr3 = distance_to_fp * scale * p3_normed# because of new_angle * 2 = angle, which creates isosceles triangle
        hp4 = - distance_to_fp / math.tan(angle) * p3_normed
        hpr4 = -p3_normed * math.cos(angle) * scale * distance_to_fp
        #decomposition onto orthogonal basis
        p1_for_scale = p1/2
        p1_p_hdir = np.dot(hdir, p1_for_scale) * hdir
        p1_p_p3 = p1 - p1_p_hdir
        #projection of basis
        vert_angle = np.arctan(np.linalg.norm(p1_p_p3)/distance_to_fp)
        p1_p_p3r = p1_p_p3*math.cos(vert_angle)/math.sin(angle-vert_angle)*scale  # some geometry
        p1_p_hdirr = p1_p_hdir * scale * math.sin(angle) *math.cos(vert_angle)/ math.sin(
            angle - vert_angle)
        p1r =  p1_p_p3r+p1_p_hdirr #  decompose vector into hp3 and hdir, project hdir with scale2(where scale2 computed from sinus theorem and scale), and hp3 with scale of (smth)

        cos_rot = np.dot(p1r,np.array((1,0)))/np.linalg.norm(p1r)
        sin_rot = math.sqrt(1-cos_rot**2)
        rot_mat = np.array([[cos_rot,sin_rot],[-sin_rot,cos_rot]])# todo maybe wrong direction

        print(np.array([hp1,hp2,hp3,hp4])+center_pos,np.array([hpr1,hpr2,hpr3,hpr4])+center_pos,rot_mat,angle)
        hpr1 = rot_mat.dot(hpr1)
        hpr2 = rot_mat.dot(hpr2)
        hpr3 = rot_mat.dot(hpr3)
        hpr4 = rot_mat.dot(hpr4)
        return cv2.findHomography(np.array([hp1,hp2,hp3,hp4])+center_pos,np.array([hpr1,hpr2,hpr3,hpr4])+center_pos+200)
        # return cv2.findHomography(np.array([hpr1,hpr2,hpr3,hpr4]),np.array([hp1,hp2,hp3,hp4])+400)
        # return cv2.getPerspectiveTransform(np.array([hp1,hp2,hp3,hp4]),np.array([hpr1,hpr2,hpr3,hpr4]))
    if True:
        hor_point1 = np.copy(hor_point1)
        hor_point2 = np.copy(hor_point2)
        hor_point1-=center_pos
        hor_point2-=center_pos
        print("hor points: ",hor_point1,hor_point2)
        p3 = calculate_p3(hor_point1,hor_point2)
        distance = find_distance_fp_to_plane_v2(hor_point1,hor_point2,p3,angle=math.pi/180*87)
        angle = angle_of_camera_v1(distance,p3)
        assert(abs(angle)>1e-5)
        print(distance,angle)
        return homography_matrix_internal(hor_point1,angle,0.4,p3,distance)
# matrix = homography_matrix(segments)

# print(white_color_img)

if False:
    # homography_matrix_v2(np.array([10.,0.]),np.array([-10.,0.]),(100,100))
    for i in range(1,5):
        center = np.array([100,100])
        offset = np.array([0,10])
        homography_matrix_v2(np.array([10.*i,0.])+center+offset,np.array([-10.*i,0.])+center+offset,center)
    # print("-"*200)
    # for i in range(1,5):#ok
    #     center = np.array([100,100])*i
    #     offset = np.array([0,10])
    #     homography_matrix_v2(np.array([10.,0.])+offset+center,np.array([-10.,0.])+offset+center,center)
    # print("-"*200)   #ok,same as previous
    # for i in range(1,5):
    #     center = np.array([100,100])*i
    #     offset = np.array([0,10])
    #     homography_matrix_v2(np.array([-10.,0.])+offset+center,np.array([10.,0.])+offset+center,center)
    # print("-"*200)   #ok, same as previous
    # for i in range(1,5):
    #     center = np.array([100,100])*i
    #     offset = np.array([10,0])
    #     homography_matrix_v2(np.array([0.,10.])+offset+center,np.array([0.,-10.])+offset+center,center)
    print("-"*200)#this test is failing because my code now works as it should and it cannot process perfectly horizontal camera position
    for i in range(1,5):
        center = np.array([100,100])
        offset = np.array([0,0])
        homography_matrix_v2(np.array([10.*i,0.])+center+offset,np.array([-10.*i,0.])+center+offset,center)
    print("-"*200)
    for i in range(-3,3):
        center = np.array([100,100])
        offset = np.array([0,10])*i
        homography_matrix_v2(np.array([100.,0.])+offset+center,np.array([-100.,0.])+offset+center,center)
    # print("-"*200)
    # for i in range(1,5): #ok
    #     center = np.array([100,100])*i
    #     offset = np.array([0,10])
    #     homography_matrix_v2(np.array([-10.,0.])+offset+center,np.array([10.,0.])+offset+center,center)
    # print("-"*200)
    # for i in range(1,5): #ok
    #     center = np.array([100,100])*i
    #     offset = np.array([0,-10])
    #     homography_matrix_v2(np.array([-10.,0.])+offset+center,np.array([10.,0.])+offset+center,center)



In [15]:

# output_2 = cv2.warpPerspective(img_tmp,matrix,(3000,3000))
output_2 = output
print(output_2.shape)
# segments = cv2.perspectiveTransform(np.array(segments).reshape((len(segments)*2,1,2)).astype(np.float32),matrix).reshape((len(segments),2,2)).astype(int)+np.array((500,500))
# print(segments)
# print(lines)
output_2 //= 5
for i in range(len(segments)):
    output_2 = cv2.line(output_2, segments[i][0], segments[i][1], 255)  # 40 kmeans is good

cv2.imshow('lanes', output_2)
cv2.waitKey(0)
cv2.destroyAllWindows()

(537, 714)
